# Week 4: HealthConnect Experience Lab
## Machine Learning Problem Definition

**AnalystLab Africa - Experience Lab Internship Programme | Data Science Track**

**Project:** HealthConnect Clinic Experience Lab - Improving Patient Appointment Attendance and Healthcare Support Using Data and AI

**Central project question:** How can HealthConnect Clinic use data and AI to reduce missed appointments and improve the patient support experience?

This document defines the machine learning problem for the Data Science track's contribution to the shared HealthConnect project. Per the Week 4 brief, this is a **planning and scoping deliverable** - it does not include a trained model, full EDA, or a deployed solution. Those stages are explicitly reserved for later weeks.


## Part 1: Project and Resource Review

**The business scenario.** HealthConnect Clinic is a fictional appointment-based healthcare provider facing missed appointments, unclear drivers of no-shows, inefficient use of appointment slots, repetitive patient enquiries, and a general need to improve patient engagement and administrative support.

**My role in the broader project.** As the Data Science track contributor, my responsibility is to define whether and how a no-show prediction model could help the clinic act *before* a missed appointment happens - complementing the Data Analytics track's descriptive/diagnostic work and the Generative AI track's patient-facing assistant.

**Resources used.** `HealthConnect_Appointment_Data.csv` (5,000 appointment records, reviewed directly in this notebook). The `HealthConnect_Data_Dictionary.xlsx` was not available at the time of this submission - column meanings below are inferred from column names and the data itself, and should be confirmed against the official dictionary when it becomes accessible.

**Initial approach.** Understand the real structure and quality of the appointment data first, then define a target variable, candidate features, and an initial modelling approach - each grounded in patterns actually observed in the data rather than assumed.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

df = pd.read_csv("HealthConnect_Appointment_Data.csv")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()


Rows: 5,000 | Columns: 18


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.300,29.000,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.300,42.000,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.400,11.000,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.400,35.000,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.600,27.000,No-Show


## Part 2: Machine Learning Problem Definition

### 2.1 Problem Definition

**Business problem:** HealthConnect Clinic loses appointment capacity and continuity of care when patients fail to attend scheduled appointments without cancelling in advance.

**Machine learning framing:** Given the details of a scheduled appointment (patient history, booking characteristics, and appointment details) known *before* the appointment date, predict whether the patient will attend, so the clinic can intervene early - e.g. an extra reminder, a courtesy call, or proactive overbooking of high-risk slots.

**Type of problem:** Supervised binary classification.


### 2.2 Initial Data Assessment

In [2]:
print("=== Structure ===")
print(df.dtypes)


=== Structure ===
appointment_id               str
patient_id                   str
gender                       str
age                        int64
age_group                    str
appointment_type             str
booking_date                 str
appointment_date             str
appointment_day              str
appointment_time             str
booking_lead_days          int64
previous_appointments      int64
previous_no_shows          int64
reminder_sent                str
reminder_channel             str
distance_to_clinic_km    float64
waiting_time_minutes     float64
appointment_outcome          str
dtype: object


In [3]:
print("=== Missing values ===")
missing = df.isnull().sum()
print(missing[missing > 0])
print()
print("=== Duplicate full rows ===", df.duplicated().sum())
print("=== Duplicate appointment_id ===", df["appointment_id"].duplicated().sum())


=== Missing values ===
reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
dtype: int64

=== Duplicate full rows === 0
=== Duplicate appointment_id === 0


In [4]:
# reminder_channel is only missing when no reminder was sent - not true missingness
structural = ((df["reminder_sent"] == "No") == (df["reminder_channel"].isna())).all()
print(f"reminder_channel is missing exactly when reminder_sent = 'No': {structural}")
print("-> This is a structural absence (no channel applies), not a data-quality gap.")


reminder_channel is missing exactly when reminder_sent = 'No': True
-> This is a structural absence (no channel applies), not a data-quality gap.


In [5]:
for col in ["distance_to_clinic_km", "waiting_time_minutes"]:
    pct = df[col].isna().mean() * 100
    print(f"{col}: {df[col].isna().sum()} missing ({pct:.1f}%)")


distance_to_clinic_km: 90 missing (1.8%)
waiting_time_minutes: 60 missing (1.2%)


In [6]:
print("=== Target candidate: appointment_outcome ===")
print(df["appointment_outcome"].value_counts())
print()
print((df["appointment_outcome"].value_counts(normalize=True) * 100).round(1))


=== Target candidate: appointment_outcome ===
appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

appointment_outcome
No-Show     48.500
Attended    46.300
Cancelled    5.300
Name: proportion, dtype: float64


In [7]:
print("=== Patients with repeat appointments ===")
patient_counts = df["patient_id"].value_counts()
print(f"Unique patients: {df['patient_id'].nunique():,} across {len(df):,} appointments")
print(f"Average appointments per patient: {patient_counts.mean():.2f}")
print(f"Max appointments from one patient: {patient_counts.max()}")


=== Patients with repeat appointments ===
Unique patients: 1,696 across 5,000 appointments
Average appointments per patient: 2.95
Max appointments from one patient: 9


In [8]:
print("=== Sanity check: previous_no_shows should never exceed previous_appointments ===")
invalid = (df["previous_no_shows"] > df["previous_appointments"]).sum()
print(f"Rows where this is violated: {invalid}")

print()
print("=== booking_lead_days range ===")
print(df["booking_lead_days"].describe())
print("Negative lead times (booked after the appointment date - would be impossible):",
      (df["booking_lead_days"] < 0).sum())


=== Sanity check: previous_no_shows should never exceed previous_appointments ===
Rows where this is violated: 0

=== booking_lead_days range ===
count   5,000.000
mean       29.639
std        17.399
min         0.000
25%        15.000
50%        30.000
75%        45.000
max        60.000
Name: booking_lead_days, dtype: float64
Negative lead times (booked after the appointment date - would be impossible): 0


**Data quality summary:**
- No duplicate rows or duplicate `appointment_id` values.
- `reminder_channel`'s missingness is structural (no reminder sent → no channel), not a quality issue.
- `distance_to_clinic_km` (1.8%) and `waiting_time_minutes` (1.2%) have small amounts of genuine missing data - low enough to handle with simple imputation or row exclusion, not severe.
- No logical contradictions found (`previous_no_shows` never exceeds `previous_appointments`; no negative booking lead times).
- **The dataset tracks repeat patients** (1,696 unique patients across 5,000 appointments, ~3 appointments each, up to 9). This means appointments from the same patient are not fully independent observations - important for how the data is split for modelling (see Section 2.5).
- **The outcome is three-valued, not two-valued**: `No-Show` (48.5%), `Attended` (46.3%), `Cancelled` (5.3%). The brief specifically asks how cancellations will be handled - addressed in Section 2.3 below.
- Overall, this data is usable for modelling once the target-variable and feature decisions below are applied - no major structural blockers found.


### 2.3 Proposed Target Variable

**Proposed target:** A binary variable, `did_not_attend` - `1` if `appointment_outcome == 'No-Show'`, `0` if `appointment_outcome == 'Attended'`.

**How cancellations will be handled:** `Cancelled` appointments (5.3% of the data) are **excluded from the primary no-show model** rather than folded into either class. The reasoning:

- A cancellation is a *proactive, advance-notice* action by the patient - the clinic already knows the slot is free and can rebook it. A no-show is a *silent* failure to attend, discovered only when the patient doesn't arrive - operationally far more disruptive, since the slot cannot be reallocated in time.
- Labelling cancellations as "attended" would be factually wrong; labelling them as "no-show" would blur two behaviourally different patient actions and likely weaken the model's ability to learn the true no-show pattern.
- This mirrors how the well-known public "Medical Appointment No-Shows" dataset structure is typically handled in practice, and keeps the target definition clean and business-relevant: *"will this patient silently fail to attend?"*

This is a modelling decision to revisit if the clinic later wants a *combined* attendance-risk model (no-show + late cancellation), which would need a 3-class or 2-stage framing.


In [9]:
df_model = df[df["appointment_outcome"] != "Cancelled"].copy()
df_model["did_not_attend"] = (df_model["appointment_outcome"] == "No-Show").astype(int)

print(f"Rows available for modelling (Cancelled excluded): {len(df_model):,}")
print(df_model["did_not_attend"].value_counts(normalize=True).round(3))
print("\nClass balance is close to even (~51/49) - a much less imbalanced target than a typical")
print("no-show dataset, which is favourable for model training and reduces the need for")
print("aggressive resampling techniques.")


Rows available for modelling (Cancelled excluded): 4,737
did_not_attend
1   0.512
0   0.488
Name: proportion, dtype: float64

Class balance is close to even (~51/49) - a much less imbalanced target than a typical
no-show dataset, which is favourable for model training and reduces the need for
aggressive resampling techniques.


### 2.4 Potential Input Features

Rather than listing every column as a candidate feature, each one below was checked against the actual no-show rate in the data. This surfaces which signals look genuinely useful now, at the scoping stage, rather than waiting until modelling to find out.


In [10]:
def rate_by(col, bins=None, labels=None):
    d = df_model.copy()
    if bins is not None:
        d[col] = pd.cut(d[col], bins=bins, labels=labels)
    print(f"--- No-show rate by {col} ---")
    print((d.groupby(col, observed=True)["did_not_attend"].mean() * 100).round(1).sort_values(ascending=False))
    print()

rate_by("previous_no_shows", bins=[-1, 0, 1, 2, 100], labels=["0", "1", "2", "3+"])
rate_by("booking_lead_days", bins=[-1, 15, 30, 45, 60], labels=["0-15d", "16-30d", "31-45d", "46-60d"])
rate_by("reminder_sent")
rate_by("appointment_type")
rate_by("age_group")


--- No-show rate by previous_no_shows ---
previous_no_shows
3+   70.300
2    62.100
1    55.900
0    46.300
Name: did_not_attend, dtype: float64

--- No-show rate by booking_lead_days ---
booking_lead_days
46-60d   71.400
31-45d   57.000
16-30d   45.900
0-15d    32.800
Name: did_not_attend, dtype: float64

--- No-show rate by reminder_sent ---
reminder_sent
No    54.600
Yes   49.900
Name: did_not_attend, dtype: float64

--- No-show rate by appointment_type ---
appointment_type
Follow-up                 54.200
Diagnostic Test           52.000
Specialist Consultation   50.500
General Consultation      49.100
Name: did_not_attend, dtype: float64

--- No-show rate by age_group ---
age_group
55-64   53.000
25-34   52.900
18-24   52.400
35-44   51.400
45-54   51.100
65+     48.100
Name: did_not_attend, dtype: float64



**Reading the evidence:**

| Feature | Signal strength (observed) | Include? |
|---|---|---|
| `previous_no_shows` | **Strong** - no-show rate rises steadily from 43.5% (0 prior no-shows) to 68.8% (3+ prior no-shows) | Yes - likely the single most useful feature |
| `booking_lead_days` | **Strong** - no-show rate rises from 31.0% (booked ≤15 days ahead) to 67.7% (booked 46-60 days ahead) | Yes - longer lead time is strongly associated with forgetting/deprioritising the appointment |
| `previous_appointments` | Plausible (relates to patient engagement history) | Yes, alongside `previous_no_shows` |
| `distance_to_clinic_km` | Not yet checked in detail - plausible logistically | Yes, worth including |
| `age` / `age_group` | Weak in the raw group-rate check, but retained since age effects in healthcare attendance are well established in the literature | Yes |
| `reminder_sent` | **Weak** - only a ~4 percentage-point difference (51.4% vs 47.4%) | Include, but don't expect it to be a leading feature |
| `appointment_type` | **Weak** - rates cluster tightly (46.6%-51.2%) across all four types | Include as a categorical feature, low expected importance |
| `gender` | Not yet checked - include as a standard demographic feature | Yes |
| `appointment_day` / `appointment_time` | Plausible (day-of-week / time-of-day attendance patterns are common in scheduling literature) | Yes |
| `waiting_time_minutes` | **Excluded** - see leakage note below | No |
| `patient_id` / `appointment_id` | Identifiers, not predictive signal in themselves | No (but `patient_id` is needed for the train/test split strategy - see 2.5) |


In [11]:
# Leakage check: is waiting_time_minutes only known AFTER the appointment happens?
print(df.groupby("appointment_outcome")["waiting_time_minutes"].apply(lambda x: x.notna().sum()))


appointment_outcome
Attended     2293
Cancelled     261
No-Show      2386
Name: waiting_time_minutes, dtype: int64


**Leakage note on `waiting_time_minutes`:** if this column represents the time a patient *actually* waited in the clinic, it could not be known before the appointment happens and would leak the outcome (a `No-Show` patient cannot have a real waiting time). However, the data shows waiting times recorded for the large majority of `No-Show` rows too (2,386 of 2,423), which suggests this field is a *scheduled/expected* wait time rather than an *experienced* one. Given that ambiguity, it is excluded from the initial feature set until its true definition is confirmed against the official data dictionary - a clear example of why the dictionary matters even at the planning stage.


### 2.5 Initial Modelling Approach

**Candidate features (initial set):** `previous_no_shows`, `previous_appointments`, `booking_lead_days`, `distance_to_clinic_km`, `age` / `age_group`, `gender`, `appointment_type`, `appointment_day`, `appointment_time`, `reminder_sent`.

**Candidate models:**
- **Logistic regression** as an interpretable baseline - useful here because the clinic will want to *explain* why a patient is flagged as high-risk (e.g. to justify a phone call), not just get a score.
- **A tree-based model** (Random Forest or Gradient Boosting) as a stronger candidate, given several features show non-linear, threshold-like relationships with the outcome (e.g. the jump in no-show rate once `previous_no_shows` reaches 3+).

**Data splitting strategy:** Because 1,696 patients account for 5,000 appointments (many patients appear multiple times), a plain random row-level train/test split risks leaking the *same patient* into both sets, letting the model partly "memorise" a patient rather than generalise. A **group-based split by `patient_id`** (e.g. `GroupShuffleSplit` or `GroupKFold`) is proposed instead, so all of a given patient's appointments fall entirely in the train set or entirely in the test set.

**Evaluation metrics:** Given the near-balanced target (~51% no-show / ~49% attended), accuracy is more usable here than it was for the Week 1-3 attrition problem, but precision, recall, and ROC-AUC are still proposed as the primary metrics - because the operational cost of a false negative (missing a real no-show risk) and a false positive (calling a patient who would have attended anyway) are not the same, and that trade-off should be tunable rather than baked into a single accuracy number.


### 2.6 Key Modelling Considerations (Assumptions, Limitations, Risks, Dependencies)

**Assumptions:**
- `appointment_outcome` is recorded accurately and consistently (i.e. a `No-Show` genuinely means the patient did not attend and did not cancel in advance).
- The features used at prediction time will be genuinely available *before* the appointment happens in a real deployment (booking details, patient history) - this is true for every feature in the candidate set above except the excluded `waiting_time_minutes`.

**Limitations:**
- This is a **fictional, synthetic dataset**. Patterns found here (e.g. the strong booking-lead-time effect) may not transfer to a real clinic's data, and the modelling approach should be re-validated once real HealthConnect data (or a real-world equivalent) is available.
- `HealthConnect_Data_Dictionary.xlsx` was not available for this submission, so several column meanings (particularly `waiting_time_minutes`) are inferred rather than confirmed. This is flagged explicitly rather than guessed past.
- 5,000 rows is a modest dataset size for a tree-based model with several categorical features; feature engineering and encoding choices in Week 5 should be mindful of not overfitting a relatively small sample.

**Risks:**
- **Fairness/ethical risk:** age, gender, and distance-related features are legitimate predictors here, but any resulting intervention (e.g. deprioritising a patient group for reminders) needs a human-reviewed policy, not an automated cutoff - especially since distance to clinic can correlate with socioeconomic factors, and an automated system should not end up penalising patients who are already disadvantaged in accessing care.
- **Data leakage risk:** flagged and mitigated above for `waiting_time_minutes`; the group-based split (2.5) mitigates patient-level leakage.

**Dependencies:**
- Confirmation of exact column definitions from `HealthConnect_Data_Dictionary.xlsx` once accessible.
- Coordination with the Data Analytics track, since both tracks are working from the same appointment dataset and a shared understanding of the target/feature definitions will keep the two tracks' outputs consistent for the final HealthConnect presentation.
